# OpenVINO Physical AI APIs to test a trained policy

### Test a previously trained Pi0.5 policy without a physical robot, then render the final rollout in MuJoCo

<img src="media/4.Simulation-only.png" width="800">

## 1) Install Notebook-Specific Dependencies

Install the shared tutorial environment from `requirements.txt` before starting JupyterLab. The next cell installs only the additional packages required by this notebook.

In [ ]:
import sys

PHYSICALAI_VERSION = "0.1.1"
PHYSICALAI_STUDIO_COMMIT = "d55d4b97ae3951c05abfe73cc59619c3e2ff71b1"
PHYSICALAI_TRAIN_EXTRA = "[libero]" if sys.platform.startswith("linux") else ""
PHYSICALAI_TRAIN_SPEC = (
    f"physicalai-train{PHYSICALAI_TRAIN_EXTRA} @ "
    "https://github.com/open-edge-platform/physical-ai-studio/archive/"
    f"{PHYSICALAI_STUDIO_COMMIT}.zip#subdirectory=library"
)

# Build the Linux EGL probes against the CMake installed in the tutorial environment.
%pip install --no-build-isolation "hf-egl-probe==1.0.2; sys_platform == 'linux'" "egl_probe==1.0.2; sys_platform == 'linux'"
%pip install physicalai=={PHYSICALAI_VERSION} "{PHYSICALAI_TRAIN_SPEC}" "mujoco>=3.2.0"

## 2) Configure the Notebook

This notebook validates a trained and OpenVINO-optimized Pi0.5 policy package without connecting to a physical robot.

By default, the model package is downloaded from the official [OpenVINO Physical AI Hugging Face collection](https://huggingface.co/collections/OpenVINO/physical-ai). The model is converted from `lerobot/pi05_libero_finetuned_v044`, whose model card references the LeRobot-format [`HuggingFaceVLA/libero`](https://huggingface.co/datasets/HuggingFaceVLA/libero) dataset.

The replay path downloads only the metadata, one episode parquet, and the corresponding videos. The documented benchmark API is newer than the current `physicalai-train==0.1.0` wheel, so the setup cell installs the Physical AI Studio library from a pinned source commit. On Linux it enables the `libero` extra and the official `LiberoBenchmark`; on Windows the simulator benchmark is disabled because its EGL probe is distributed as source and requires a native Linux EGL toolchain. Model loading, replay validation, and the standalone MuJoCo visualization remain available on Windows.

Useful environment variables:
- `PHYSICALAI_PI05_EXPORT_DIR`
- `PHYSICALAI_PI05_REPLAY_DATASET`
- `PHYSICALAI_PI05_REPLAY_EPISODE`
- `PHYSICALAI_PI05_TASK`
- `PHYSICALAI_ASSETS_DIR`

In [ ]:
# ============================================================
# USER PARAMETERS - Edit these before running if needed.
# ============================================================

MODEL_REPO_ID = "OpenVINO/pi05-libero-fp16-ov"
DATASET_REPO_ID = "HuggingFaceVLA/libero"
DATASET_NAME = ""

from pathlib import Path
import os
import sys

# MuJoCo must select its headless backend before PhysicalAI imports the simulator.
if sys.platform.startswith("linux"):
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") != "egl":
        raise RuntimeError("Restart the kernel before running this notebook so MuJoCo can use EGL.")
    os.environ["MUJOCO_GL"] = "egl"

import openvino as ov
import openvino_tokenizers  # Registers tokenizer custom ops used by tokenizer.xml.
from IPython.display import display
from physicalai.benchmark.gyms import LiberoBenchmark
from physicalai.inference import InferenceModel

WORKSPACE = Path.cwd().resolve()
if "HELPER_DIR" not in globals():
    HELPER_DIR = WORKSPACE
sys.path.append(str(Path(HELPER_DIR).resolve()))
sys.path.append(str((WORKSPACE / "notebooks").resolve()))

from physicalai_pi05_helper import (  # noqa: E402
    download_pi05_package,
    openvino_config_for_device,
    prepare_replay_episode,
    run_mujoco_visualization,
    run_replay_visualization,
)

TASK_TEXT = os.environ.get("PHYSICALAI_PI05_TASK")
ASSETS_DIR = Path(os.environ.get("PHYSICALAI_ASSETS_DIR", WORKSPACE / "physicalai_assets")).expanduser().resolve()

MODEL_DIR = Path(os.environ["PHYSICALAI_PI05_EXPORT_DIR"]).expanduser().resolve() if os.environ.get("PHYSICALAI_PI05_EXPORT_DIR") else None
REPLAY_DATASET_DIR = Path(os.environ["PHYSICALAI_PI05_REPLAY_DATASET"]).expanduser().resolve() if os.environ.get("PHYSICALAI_PI05_REPLAY_DATASET") else None
REPLAY_EPISODE_ID = int(os.environ.get("PHYSICALAI_PI05_REPLAY_EPISODE", "0"))
RUN_REPLAY_VALIDATION = True
RUN_LIBERO_BENCHMARK = sys.platform.startswith("linux")

# Quick benchmark defaults: one LIBERO-10 task and one episode.
# Set LIBERO_TASK_IDS = None and LIBERO_NUM_EPISODES = 20 for the full suite.
LIBERO_TASK_SUITE = "libero_10"
LIBERO_TASK_IDS = [0]
LIBERO_NUM_EPISODES = 1
LIBERO_MAX_STEPS = None
LIBERO_SEED = 42
LIBERO_RECORD_MODE = "failures"

CACHE_DIR = ASSETS_DIR / "cache" / "pi05_openvino"
VIS_DIR = ASSETS_DIR / "visualizations" / "pi05_without_robot"
LIBERO_VIDEO_DIR = VIS_DIR / "libero_benchmark_videos"
LIBERO_RESULTS_PATH = VIS_DIR / "libero_benchmark_results.json"

core = ov.Core()
print("OpenVINO:", ov.__version__)
print("Available devices:", core.available_devices)
print("Replay dataset:", DATASET_REPO_ID or REPLAY_DATASET_DIR)
print("Replay validation:", "enabled" if RUN_REPLAY_VALIDATION else "disabled")
print("LIBERO benchmark:", "enabled" if RUN_LIBERO_BENCHMARK else "disabled")

## 3) Download the Pi0.5 OpenVINO Policy Package

A PhysicalAI policy package contains the OpenVINO intermediate representation files (`pi05.xml`, `pi05.bin`), tokenizer artifacts, `manifest.json`, and processor metadata.

In [ ]:
MODEL_DIR = download_pi05_package(MODEL_REPO_ID, ASSETS_DIR, MODEL_DIR)
print("[DONE] PhysicalAI OpenVINO package is ready.")

## 4) Download and Prepare One LIBERO Replay Episode

For a fast validation pass, the notebook downloads dataset metadata, one episode parquet, and the corresponding camera videos from `HuggingFaceVLA/libero`. This is enough to build the same observation dictionary shape used by the Pi0.5 OpenVINO package.

In [ ]:
replay = None
if RUN_REPLAY_VALIDATION:
    replay = prepare_replay_episode(
        repo_id=DATASET_REPO_ID,
        dataset_name=DATASET_NAME,
        assets_dir=ASSETS_DIR,
        episode_id=REPLAY_EPISODE_ID,
        dataset_dir=REPLAY_DATASET_DIR,
    )
    if TASK_TEXT is None:
        TASK_TEXT = replay.task or "pick up the object"
    print(f"[DONE] Replay episode {replay.episode_id}: {len(replay.episode_df)} frames at {replay.fps:.1f} FPS")
    print("[DONE] Replay image keys:", replay.image_keys)
    print("[DONE] Task:", TASK_TEXT)
else:
    TASK_TEXT = TASK_TEXT or "pick up the object"
    print("[SKIP] Replay validation disabled. The final MuJoCo cell will render a scripted pick-and-place trajectory.")

## 5) Select an OpenVINO Device


In [ ]:
import ipywidgets as widgets

device_options = list(core.available_devices)
default_device = "GPU" if "GPU" in device_options else "CPU"
TARGET_DEVICE = widgets.Dropdown(
    options=device_options,
    value=default_device if default_device in device_options else device_options[0],
    description="Device:",
)
display(TARGET_DEVICE)


## 6) Load and Benchmark with PhysicalAI Runtime

`InferenceModel.load()` loads the exported OpenVINO policy package on the selected device. The official [`LiberoBenchmark`](https://github.com/open-edge-platform/physical-ai-studio/tree/main/library#benchmark) API then evaluates that deployment model in standardized LIBERO simulation episodes.

The default configuration is a functional smoke benchmark with one task and one episode. It reports simulation success rate, reward, episode length, and FPS, and writes the complete benchmark result to JSON. Use all task IDs and 20 episodes per task for the full LIBERO-10 benchmark.

In [ ]:
selected_device = TARGET_DEVICE.value
selected_result = {"device": selected_device}
benchmark_results = None

try:
    physicalai_model = InferenceModel.load(
        MODEL_DIR,
        backend="openvino",
        device=selected_device,
        **openvino_config_for_device(CACHE_DIR),
    )
except RuntimeError as exc:
    if selected_device == "CPU":
        raise
    print(f"[WARN] OpenVINO load failed on {selected_device}: {exc}")
    print("[INFO] Falling back to CPU so the notebook can continue.")
    selected_device = "CPU"
    selected_result["device"] = selected_device
    physicalai_model = InferenceModel.load(
        MODEL_DIR,
        backend="openvino",
        device=selected_device,
        **openvino_config_for_device(CACHE_DIR),
    )

print(f"[DONE] Loaded PhysicalAI Pi0.5 OpenVINO package on {selected_device}")

if RUN_LIBERO_BENCHMARK:
    libero_benchmark = LiberoBenchmark(
        task_suite=LIBERO_TASK_SUITE,
        task_ids=LIBERO_TASK_IDS,
        num_episodes=LIBERO_NUM_EPISODES,
        max_steps=LIBERO_MAX_STEPS,
        seed=LIBERO_SEED,
        video_dir=LIBERO_VIDEO_DIR,
        record_mode=LIBERO_RECORD_MODE,
    )
    print("[INFO] Running", libero_benchmark)
    benchmark_results = libero_benchmark.evaluate(physicalai_model)
    print(benchmark_results.summary())
    saved_results = benchmark_results.to_json(LIBERO_RESULTS_PATH)
    selected_result.update(
        {
            "success_rate": benchmark_results.aggregate_success_rate,
            "avg_reward": benchmark_results.aggregate_reward,
            "avg_episode_length": benchmark_results.aggregate_episode_length,
            "avg_fps": benchmark_results.aggregate_fps,
        }
    )
    print("[DONE] Saved LIBERO benchmark results:", saved_results)
else:
    print("[SKIP] LIBERO benchmark disabled; model loading was still validated.")

## 7) Optional Replay Visualization

When replay validation is enabled, this cell compares Pi0.5 OpenVINO actions with recorded expert actions while showing both camera views. The MAE is a domain-match signal, not an OpenVINO numerical correctness test.

In [ ]:
replay_result = None
if replay is not None:
    replay_result = run_replay_visualization(
        model=physicalai_model,
        model_dir=MODEL_DIR,
        replay=replay,
        task=TASK_TEXT,
        device=selected_result["device"],
        cache_dir=CACHE_DIR,
        output_dir=VIS_DIR,
        max_rendered_frames=120,
        render_stride=3,
    )

    print("[RESULT] Replay steps:", replay_result["steps"])
    print("[RESULT] Rendered frames:", replay_result["rendered_frames"])
    print("[RESULT] Avg select_action latency ms:", replay_result["avg_select_action_ms"])
    print("[RESULT] Avg MAE vs expert action:", replay_result["avg_mae"])
    print("[RESULT] Per-joint MAE:", replay_result["per_joint_mae"])
    print("[INTERPRETATION]", replay_result["interpretation"])
    print("[DONE] Saved replay GIF:", replay_result["gif_path"])
    display(replay_result["gif"])
else:
    print("[SKIP] Replay visualization skipped because no replay dataset is configured.")

## 8) MuJoCo Pick-and-Place Visualization

This final cell renders a lightweight MuJoCo pick-and-place scene so users can see a robotless simulation result. The official OpenVINO model is converted from a LIBERO/Panda policy, while this notebook uses a compact built-in MuJoCo scene for fast visualization. Because the LIBERO action space is not the same as this toy arm's joint-control space, the MuJoCo scene uses a scripted pick-and-place trajectory instead of directly driving the arm with the model's predicted action tensor.

Use the replay section above for actual OpenVINO policy inference, latency, and action-vs-expert validation. Use this MuJoCo section as a visual smoke test that the notebook can produce a simulation-style result without a physical robot.

In [ ]:
mujoco_result = run_mujoco_visualization(
    actions=None,
    output_dir=VIS_DIR,
    source="scripted MuJoCo pick-and-place trajectory",
    max_rendered_frames=180,
)

print("[RESULT] MuJoCo frames:", mujoco_result["frames"])
print("[RESULT] MuJoCo source:", mujoco_result["source"])
print("[DONE] Saved MuJoCo GIF:", mujoco_result["gif_path"])
display(mujoco_result["gif"])
